In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2
%matplotlib inline


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

from typing import List

import mne
from mne.io import read_raw_eeglab
from mne_connectivity import spectral_connectivity_epochs
from mne import make_forward_solution, setup_source_space, setup_volume_source_space
from mne.datasets import sample
from mne.io import read_raw_fif
from mne.minimum_norm import apply_inverse_epochs, make_inverse_operator
from mne.viz import circular_layout
from mne_connectivity import spectral_connectivity_epochs
from mne_connectivity.viz import plot_connectivity_circle

import random
import pickle
import logging

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)


In [3]:
def load_eeg_data(
    file_path,
    eog: tuple = (),
    verbose: bool = True,
    picks: List[str] = None,
    apply_csd=False,
):
    raw = read_raw_eeglab(
        input_fname=file_path,
        eog=eog,
        preload=True,
        montage_units="mm",
        verbose=verbose,
    )

    print(f"{raw.get_data().shape = }")
    raw = raw.pick(picks)
    print(f"{raw.get_data().shape = }")
    print(f"{raw.ch_names = }")
    raw.rename_channels(
        mapping={
            "FPZ": "Fpz",
            "OZ": "Oz",
            "FP1": "Fp1",
            "FP2": "Fp2",
            "FZ": "Fz",
            "FCZ": "FCz",
            "CPZ": "CPz",
            "CZ": "Cz",
            "PZ": "Pz",
            "POZ": "POz",
        }
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    raw.set_montage(montage)

    raw = raw.crop(tmin=10, tmax=raw.times[-1] - 10)
    raw = raw.resample(256, npad="auto")  # Change it to 500 sampling rate

    # apply csd to remove noise
    if apply_csd:
        raw = mne.preprocessing.compute_current_source_density(raw)

    return raw


def load_metadata(filepath, condition_number=1):
    """Load and preprocess metadata."""
    metadata = pd.read_csv(filepath)
    metadata["channels"] = metadata["channels"].apply(eval)
    metadata = metadata[
        (metadata["condition_number"] == condition_number)
        & (~metadata["filename"].str.contains("COG"))
    ]
    return metadata


def preprocess_raw_data(raw: mne.io.Raw, smallers_duration: float):
    if raw.times[-1] <= smallers_duration:
        return raw

    total_duration = raw.times[-1]  # in seconds
    middle_duration = int(total_duration / 2)
    start = middle_duration - int(smallers_duration / 2)
    end = middle_duration + int(smallers_duration / 2)

    # print(f"{start = }")
    # print(f"{end = }")

    raw = raw.crop(tmin=start, tmax=end)

    return raw


def create_epochs_from_raw(
    raw, epoch_duration=2.0, overlap=0.0, tmin=0, tmax=None, baseline=None
):
    """
    Create epochs from raw MNE data.

    Parameters:
    -----------
    raw : mne.io.Raw
        The raw MNE data.
    epoch_duration : float, optional
        Duration of each epoch in seconds. Default is 2.0.
    overlap : float, optional
        Overlap between epochs in seconds. Default is 0.0 (no overlap).
    tmin : float, optional
        Start time of the epoch in seconds. Default is 0.
    tmax : float, optional
        End time of the epoch in seconds. If None, it will be set to epoch_duration.
    baseline : tuple or None, optional
        The baseline to apply. If None, no baseline is applied.

    Returns:
    --------
    epochs : mne.Epochs
        The created epochs.
    """
    # Create fixed-length events
    events = mne.make_fixed_length_events(raw, duration=epoch_duration, overlap=overlap)

    # If tmax is not specified, set it to epoch_duration
    if tmax is None:
        tmax = epoch_duration

    # Create epochs
    epochs = mne.Epochs(
        raw, events, tmin=tmin, tmax=tmax, baseline=baseline, preload=True
    )

    return epochs


def calculate_connectivity(
    epochs,
    method="wpli",
    fmin=8,
    fmax=13,
    faverage=True,
    mode="multitaper",
    mt_adaptive=False,
    n_jobs=1,
):
    """
    Calculate connectivity using spectral_connectivity_epochs.

    Parameters:
    -----------
    epochs : mne.Epochs
        The epochs to use for connectivity calculation.
    method : str, optional
        Connectivity method to use. Default is 'wpli'.
    fmin : float, optional
        Minimum frequency of interest. Default is 8 (lower bound of alpha band).
    fmax : float, optional
        Maximum frequency of interest. Default is 13 (upper bound of alpha band).
    faverage : bool, optional
        Whether to average across frequencies. Default is True.
    mode : str, optional
        Spectrum estimation mode. Default is 'multitaper'.
    mt_adaptive : bool, optional
        Whether to use adaptive weights for multitaper method. Default is False.
    n_jobs : int, optional
        Number of jobs to run in parallel. Default is 1.

    Returns:
    --------
    con : ndarray
        Connectivity matrix.
    freqs : ndarray
        Frequencies used in the calculation.
    times : ndarray
        Time points used in the calculation.
    n_epochs : int
        Number of epochs used.
    n_tapers : int
        Number of tapers used.
    """
    from mne_connectivity import spectral_connectivity_epochs

    sfreq = epochs.info["sfreq"]

    con = spectral_connectivity_epochs(
        epochs,
        method=method,
        mode=mode,
        sfreq=sfreq,
        fmin=fmin,
        fmax=fmax,
        faverage=faverage,
        tmin=epochs.tmin,
        mt_adaptive=mt_adaptive,
        n_jobs=n_jobs,
    )

    return con


def plot_connectivity(connectivity, method: str = "wpli"):
    """
    Plot connectivity matrix.

    Parameters:
    -----------
    connectivity : ndarray
        The connectivity matrix to plot. Should be a 3D array where the first two dimensions
        represent channels and the third dimension is 1 (as returned by spectral_connectivity_epochs).
    method : str, optional
        The connectivity method used, for the plot title. Default is "wpli".

    Returns:
    --------
    None. Displays the plot.
    """
    if not isinstance(connectivity, np.ndarray):
        connectivity = connectivity.get_data("dense")[:, :, 0]

    fig, ax = plt.subplots(figsize=(10, 8))

    im = ax.imshow(
        connectivity,
        vmin=0,
        vmax=1,
        cmap="tab20c",
        interpolation="nearest",
    )

    ax.set_title(f"{method.upper()} Connectivity")
    ax.set_ylabel("Channels")
    ax.set_xlabel("Channels")

    # Add colorbar
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(method.upper())

    plt.tight_layout()

    return fig


def detect_channel_sides(channels):
    """
    Detect whether each channel is on the left or right side.

    Args:
    channels (list): List of channel names.

    Returns:
    dict: Dictionary with channel names as keys and 'Left' or 'Right' as values.
    """

    def get_side(channel):
        numeric_part = "".join(filter(str.isdigit, channel))
        if numeric_part and int(numeric_part) % 2 == 0:
            return "Right"
        return "Left"

    return {channel: get_side(channel) for channel in channels}


def assign_channel_colors(channels_side, left_color, right_color):
    """
    Assign colors to channels based on their side.

    Args:
    channels_side (dict): Dictionary of channels and their sides.
    left_color (tuple): RGBA color for left channels.
    right_color (tuple): RGBA color for right channels.

    Returns:
    dict: Dictionary with channel names as keys and color tuples as values.
    """
    return {
        channel: left_color if side == "Left" else right_color
        for channel, side in channels_side.items()
    }


def assign_channel_5_colors():
    red = (0.458, 0.0, 0.0, 1.0)
    green = (0.0, 1.0, 0.0, 1.0)
    blue = (0.0, 0.0, 0.999, 1.0)
    purple = (0.5, 0.0, 0.5, 1.0)
    orange = (1.0, 0.647, 0.0, 1.0)

    channel_color = {
        # Frontal (Red)
        "AF3": red,
        "AF4": red,
        "F1": red,
        "F2": red,
        "F3": red,
        "F4": red,
        "F5": red,
        "F6": red,
        "F7": red,
        "F8": red,
        "FC1": red,
        "FC2": red,
        "FC3": red,
        "FC4": red,
        "FC5": red,
        "FC6": red,
        "FCZ": red,
        "FP1": red,
        "FP2": red,
        "FPZ": red,
        "FT7": red,
        "FT8": red,
        "FZ": red,
        # Central (Green)
        "C1": green,
        "C2": green,
        "C3": green,
        "C4": green,
        "C5": green,
        "C6": green,
        "CZ": green,
        # Parietal (Blue)
        "CP1": blue,
        "CP2": blue,
        "CP3": blue,
        "CP4": blue,
        "CP5": blue,
        "CP6": blue,
        "CPZ": blue,
        "P1": blue,
        "P2": blue,
        "P3": blue,
        "P4": blue,
        "P5": blue,
        "P6": blue,
        "P7": blue,
        "P8": blue,
        "PZ": blue,
        # Occipital (Purple)
        "O1": purple,
        "O2": purple,
        "OZ": purple,
        "PO3": purple,
        "PO4": purple,
        "PO5": purple,
        "PO6": purple,
        "PO7": purple,
        "PO8": purple,
        "POZ": purple,
        # Temporal (Orange)
        "T7": orange,
        "T8": orange,
        "TP7": orange,
        "TP8": orange,
    }

    return channel_color


def get_channel_5_colors():
    return [
        "AF3",
        "F1",
        "F3",
        "F5",
        "F7",
        "FC1",
        "FC3",
        "FC5",
        "FCZ",
        "FP1",
        "FPZ",
        "FT7",
        "FZ",
        "C1",
        "C3",
        "C5",
        "CZ",
        "T7",
        "CP1",
        "CP3",
        "CP5",
        "CPZ",
        "P1",
        "P3",
        "P5",
        "P7",
        "PZ",
        "PO3",
        "PO5",
        "PO7",
        "POZ",
        "O1",
        "OZ",
        "TP7",
        "TP8",
        "T8",
        "PO8",
        "PO6",
        "PO4",
        "O2",
        "P8",
        "P6",
        "P4",
        "P2",
        "CP6",
        "CP4",
        "CP2",
        "C6",
        "C4",
        "C2",
        "FT8",
        "FP2",
        "FC6",
        "FC4",
        "FC2",
        "F8",
        "F6",
        "F4",
        "F2",
        "AF4",
    ]


def prepare_circular_layout(label_names):
    """
    Prepare the circular layout for the connectivity plot.

    Args:
    label_names (list): List of channel names.

    Returns:
    tuple: Node angles and node order for the circular layout.
    """
    n_channels = len(label_names)
    node_angles = circular_layout(label_names, label_names, start_pos=90)
    node_order = label_names
    return node_angles, node_order


def get_circular_plot_requirements(label_names):
    """
    Get the requirements for a circular connectivity plot with left channels first.

    Args:
    label_names (list): List of channel names.

    Returns:
    tuple: Node angles, node order, and node colors for the circular plot.
    """
    # Define colors
    left_color = (0.0, 0.0, 0.999, 1.0)
    right_color = (0.458, 0.0, 0.0, 1.0)

    # Detect channel sides
    channels_side = detect_channel_sides(label_names)

    # Sort channels: left first, then right
    left_channels = [ch for ch, side in channels_side.items() if side == "Left"]
    right_channels = [ch for ch, side in channels_side.items() if side == "Right"]
    # sorted_channels = left_channels + right_channels

    sorted_channels = get_channel_5_colors()

    # Assign colors to channels
    # channel_color = assign_channel_colors(channels_side, left_color, right_color)
    channel_color = assign_channel_5_colors()

    # Prepare circular layout with sorted channels
    node_angles, _ = prepare_circular_layout(list(channel_color.keys()))

    # Get node colors in the same order as sorted_channels
    node_colors = [channel_color[channel] for channel in sorted_channels]

    return node_angles, sorted_channels, node_colors


def plot_circular_connectivity(
    connectivity,
    stage,
    label_names,
    node_angles,
    node_colors,
    method="wpli",
    is_save=False,
    n_lines=20,
    figsize=(8, 8),
):
    """
    Plot circular connectivity diagram.

    Args:
    connectivity (mne.Connectivity): The connectivity object.
    stage (str): The stage of the experiment (e.g., 'pre', 'during', 'post').
    label_names (list): List of channel names.
    node_angles (dict): Dictionary of node angles for the circular plot.
    node_colors (list): List of colors for each node.
    method (str, optional): Connectivity method used. Defaults to "wpli".
    is_save (bool, optional): Whether to save the figure. Defaults to False.
    n_lines (int, optional): Number of connections to draw. Defaults to 20.
    figsize (tuple, optional): Figure size. Defaults to (8, 8).

    Returns:
    matplotlib.figure.Figure: The created figure.
    """
    # Create figure
    fig, ax = plt.subplots(
        figsize=figsize, facecolor="black", subplot_kw=dict(polar=True)
    )

    # Plot connectivity circle
    plot_connectivity_circle(
        con=connectivity.get_data("dense")[:, :, 0],
        node_names=label_names,
        n_lines=n_lines,
        node_angles=node_angles,
        node_colors=node_colors,
        title=f"All-to-All Connectivity {stage} stimuli Condition ({method})",
        ax=ax,
    )

    # Adjust layout
    fig.tight_layout()

    # Save figure if requested
    if is_save:
        filename = f"{method}_{stage}.png"
        fig.savefig(filename, dpi=300)
        print(f"Figure saved as {filename}")

    return fig


def save_connectivity(con, filepath):
    """Save connectivity data."""
    with open(filepath, "wb") as f:
        pickle.dump(con, f)


def process_patient_data(
    metadata, patient, stage_no, smallest_duration, output_dir, apply_csd, **kwargs
):
    """Process data for a single patient and stage."""
    patient_data = metadata[
        (metadata["patient"] == patient) & (metadata["stage_no"] == stage_no)
    ]
    if patient_data.empty:
        logging.warning(f"No data found for patient {patient}, stage {stage_no}")
        return

    patient_data = patient_data.iloc[0]
    filepath, channels = patient_data[["filepath", "channels"]]

    raw = load_eeg_data(filepath, picks=channels, apply_csd=apply_csd)
    raw = preprocess_raw_data(raw, smallest_duration)
    epochs = create_epochs_from_raw(raw)
    con = calculate_connectivity(epochs=epochs, **kwargs)

    # Save connectivity data
    if stage_no == 1:
        stage_name = "pre"
    elif stage_no == 2:
        stage_name = "during"
    else:
        stage_name = "post"

    save_filepath = os.path.join(output_dir, f"{patient}_{stage_name}_connectivity.pkl")
    save_connectivity(con, save_filepath)

    # Plot and save connectivity
    plt.figure(figsize=(10, 8))
    fig = plot_connectivity(con, method="wpli")
    plt.savefig(
        os.path.join(output_dir, f"{patient}_{stage_name}_connectivity_plot.png")
    )
    plt.close()

    # Plot and save circular connectivity
    node_angles, node_order, node_colors = get_circular_plot_requirements(channels)
    fig = plot_circular_connectivity(
        con, stage_name, node_order, node_angles, node_colors
    )
    fig.savefig(os.path.join(output_dir, f"{patient}_{stage_name}_circular_plot.png"))
    plt.close(fig)


def get_patient_epoch_data(metadata, patient, stage_no, smallest_duration):
    """Process data for a single patient and stage."""
    patient_data = metadata[
        (metadata["patient"] == patient) & (metadata["stage_no"] == stage_no)
    ]
    if patient_data.empty:
        logging.warning(f"No data found for patient {patient}, stage {stage_no}")
        return

    patient_data = patient_data.iloc[0]
    filepath, channels = patient_data[["filepath", "channels"]]

    raw = load_eeg_data(filepath, picks=channels)
    raw = preprocess_raw_data(raw, smallest_duration)
    epochs = create_epochs_from_raw(raw)
    return epochs


In [ ]:
metadata_path = "../metadata.csv"
metadata = pd.read_csv(metadata_path)
metadata["channels"] = metadata["channels"].apply(lambda x: eval(x))

metadata = metadata[metadata["condition_number"] == 1]
metadata = metadata[metadata["stage_no"] != 2]
metadata = metadata[~metadata["filename"].str.contains("COG")]

smallest_duration = metadata["duration"].min()
print(f"{smallest_duration = }")


In [ ]:
stage_no = 1  # pre
patient = 2

filepath, channels = metadata[
    (metadata["patient"] == patient) & (metadata["stage_no"] == stage_no)
].iloc[0][["filepath", "channels"]]

raw = load_eeg_data(filepath, picks=channels)
raw = preprocess_raw_data(raw, smallest_duration)

# Create epochs
epochs = create_epochs_from_raw(
    raw, epoch_duration=2.0, overlap=0.0, tmin=0, tmax=2, baseline=None
)

# Calculate connectivity
con = calculate_connectivity(
    epochs,
    method="wpli",
    fmin=4,
    fmax=50,
    faverage=True,
    mode="multitaper",
    mt_adaptive=False,
    n_jobs=1,
)


In [ ]:
plot_connectivity(con, method="wpli")


In [ ]:
node_angles, node_order, node_colors = get_circular_plot_requirements(channels)
fig = plot_circular_connectivity(con, "pre", node_order, node_angles, node_colors)
plt.show()


In [ ]:
def main():
    metadata_path = "../metadata.csv"
    for condition_no in [1, 2, 3, 4, 5, 6]:  # Process conditions 1, 2, and 3
        for fmin, fmax in [
            (1, 4),
            (4, 8),
            (8, 12),
            (13, 30),
            (30, 50),
            (1, 120),
            (65, 120),
        ]:
            metadata = load_metadata(metadata_path, condition_number=condition_no)
            smallest_duration = metadata["duration"].min()
            logging.info(f"Smallest duration: {smallest_duration}")
            for patient in range(2, 22):  # Process patients 2 to 21
                output_dir = f"csd/filters/{fmin}_{fmax}/conditions/{condition_no}/results/{patient}"
                os.makedirs(output_dir, exist_ok=True)
                logging.info(f"Processing data for patient {patient}")

                for stage_no in [1, 2, 3]:  # pre and post
                    try:
                        process_patient_data(
                            metadata,
                            patient,
                            stage_no,
                            smallest_duration,
                            output_dir,
                            apply_csd=True,  # Apply CSD
                            fmin=fmin,
                            fmax=fmax,
                        )
                        logging.info(
                            f"Processed data for patient {patient}, stage {stage_no}"
                        )
                    except Exception as e:
                        logging.error(
                            f"Error processing data for patient {patient}, stage {stage_no}: {e}"
                        )


if __name__ == "__main__":
    main()


In [ ]:
def main():
    metadata_path = "../metadata.csv"
    metadata = load_metadata(metadata_path)
    smallest_duration = metadata["duration"].min()
    logging.info(f"Smallest duration: {smallest_duration}")

    total_epochs = []
    for patient in range(2, 22):  # Process patients 2 to 21
        logging.info(f"Processing data for patient {patient}")

        temp = []
        for stage_no in [1, 3]:  # pre and post
            try:
                epochs = get_patient_epoch_data(
                    metadata, patient, stage_no, smallest_duration
                )

                temp.append(epochs)  # pre then post
                logging.info(
                    f"Processed data for patient {patient}, stage {stage_no}, no. of epoch {epochs.shape}"
                )
            except Exception as e:
                logging.error(
                    f"Error processing data for patient {patient}, stage {stage_no}: {e}"
                )

        total_epochs.append(temp)

    print(f"{len(total_epochs) = }")


if __name__ == "__main__":
    main()
